In [1]:
import pandas as pd
import numpy as np
import datetime
from datetime import datetime
from datetime import date

In [2]:
fp = 'monthly_data.dat'
openET = pd.read_csv(fp, sep = '\t', header = 1)
openET

,Site ID,geeSEBAL,PT-JPL,SSEBop,SIMS,eeMETRIC,DisALEXI,Ensemble,DATE
0,US-Blk,16.2,12.1,20.7,NaN,9.2,12.5,14.2,2004-02-01
1,US-Blk,6.2,3.7,35.7,NaN,10.9,13.1,8.5,2004-12-01
2,US-Blk,10.0,6.2,19.0,NaN,4.9,15.8,11.2,2005-01-01
3,US-Blk,11.8,15.1,26.8,NaN,9.5,26.4,17.9,2005-02-01
4,US-Blk,40.0,29.5,41.5,NaN,22.0,55.2,37.7,2005-03-01
...,...,...,...,...,...,...,...,...,...
4095,VR,159.2,165.3,200.3,NaN,201.9,186.7,182.7,2003-07-01
4096,VR,163.8,165.0,181.5,NaN,188.4,158.4,167.2,2003-08-01
4097,VR,111.4,136.1,154.9,NaN,110.8,115.8,118.5,2003-09-01
4098,VR,22.4,61.0,24.6,NaN,11.2,29.9,22.0,2003-12-01


In [3]:
site_id = openET["Site ID"].values
dates = openET["DATE"].values
openET_sites = openET['Site ID'].unique()

In [4]:
site_time = np.ndarray(shape = (len(openET_sites),5), dtype = 'O')
site_time[:,0] = openET_sites


for i in range(len(openET)-1):
  for j in range(len(openET_sites)):
    if site_id[i] == openET_sites[j] and site_id[i-1] != openET_sites[j]:
      site_time[j,1] = dates[i]
    if site_id[i] == openET_sites[j] and site_id[i+1] != openET_sites[j]:
      site_time[j,2] = dates[i]
    difference = np.datetime64(site_time[j,2]) - np.datetime64(site_time[j,1])
    site_time[j,3] = (difference.astype('timedelta64[Y]').item())
    site_time[124,2] = '2004-05-01'
site_time

array([['US-Blk', '2004-02-01', '2008-03-01', 4, None],
       ['US-Blo', '2004-03-01', '2007-09-01', 3, None],
       ['US-CZ3', '2009-07-01', '2012-06-01', 2, None],
       ['US-Fmf', '2005-09-01', '2010-11-01', 5, None],
       ['US-Fuf', '2005-10-01', '2010-11-01', 5, None],
       ['US-GLE', '2001-06-01', '2017-11-01', 16, None],
       ['US-Me2', '2002-03-01', '2020-06-01', 18, None],
       ['US-Me5', '2001-02-01', '2002-12-01', 1, None],
       ['US-Me6', '2011-06-01', '2020-06-01', 9, None],
       ['US-NC2', '2017-04-01', '2019-06-01', 2, None],
       ['US-NC3', '2017-02-01', '2018-03-01', 1, None],
       ['US-NR1', '2001-02-01', '2019-12-01', 18, None],
       ['US-SP2', '2001-02-01', '2003-12-01', 2, None],
       ['US-SP3', '2001-02-01', '2008-11-01', 7, None],
       ['ALARC2_Smith6', '2018-02-01', '2018-05-01', 0, None],
       ['Almond_High', '2016-11-01', '2019-09-01', 2, None],
       ['Almond_Low', '2016-10-01', '2019-09-01', 2, None],
       ['Almond_Med', '2016-1

In [5]:
site_time[:,4].astype('F')
site_time[:,4] = 0
for i in range(len(site_time)):
  for j in range(len(openET)):
    if site_id[j] == site_time[i,0]:
      site_time[i,4] = site_time[i,4] + 1
columns = ('Site ID', 'Start Date', 'End Date', '# of Years', '# of Occurrences')
df = pd.DataFrame(data=site_time, columns = columns)

In [6]:
min_years = 10
long_sites = site_time[site_time[:,3] > min_years]
df = pd.DataFrame(data=long_sites, columns = columns)
print(df)

   Site ID  Start Date    End Date # of Years # of Occurrences
0   US-GLE  2001-06-01  2017-11-01         16              103
1   US-Me2  2002-03-01  2020-06-01         18              104
2   US-NR1  2001-02-01  2019-12-01         18              184
3   US-ARM  2003-01-01  2020-08-01         17               86
4   US-Ne1  2001-07-01  2019-12-01         18              203
5   US-Ne2  2001-07-01  2019-12-01         18              202
6   US-Ne3  2001-07-01  2019-12-01         18              197
7   US-SRG  2008-06-01  2020-05-01         11               78
8   US-Var  2003-11-01  2020-07-01         16              200
9   US-Wkg  2004-06-01  2020-05-01         15               48
10  US-MMS  2004-04-01  2020-06-01         16              111
11  US-WCr  2006-06-01  2019-06-01         12               12
12  US-SRM  2004-02-01  2020-05-01         16               96
13  US-CMW  2001-04-01  2019-10-01         18              121


In [9]:
site_year_counts = {}
years = range(2001, 2021)

for site in long_sites[:, 0]:
  site_year_counts[site] = {}
  for year in years:
    site_year_counts[site][year] = 0

for i in range(len(openET)):
  site = site_id[i]
  if site in long_sites[:, 0]:
    year = datetime.strptime(dates[i], '%Y-%m-%d').year
    if year in years:
      site_year_counts[site][year] += 1

df_year_counts = pd.DataFrame.from_dict(site_year_counts, orient='index', columns=years).fillna(0)
df_year_counts.to_latex("site_counts.txt")
df_year_counts

,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020
US-GLE,1,0,0,3,7,9,9,7,6,8,7,9,7,8,7,8,7,0,0,0
US-Me2,0,8,0,2,3,2,1,2,5,4,5,6,8,5,9,10,10,10,8,6
US-NR1,10,12,10,12,9,11,7,8,8,9,7,11,8,8,11,9,12,12,10,0
US-ARM,0,0,5,4,5,12,4,5,5,6,8,9,2,7,3,2,3,2,1,3
US-Ne1,6,12,12,12,12,12,12,12,12,12,12,12,12,12,4,4,10,11,12,0
US-Ne2,6,12,12,12,12,12,12,12,12,12,12,12,12,12,5,3,10,10,12,0
US-Ne3,6,12,12,12,12,12,12,12,12,12,12,12,12,12,0,3,10,10,12,0
US-SRG,0,0,0,0,0,0,0,4,7,5,8,4,3,9,5,9,7,9,5,3
US-Var,0,0,2,12,12,12,12,12,12,11,12,12,12,12,12,12,12,12,12,7
US-Wkg,0,0,0,1,6,2,5,3,2,3,3,3,2,2,3,1,4,4,2,2
